# A3: Architecture Comparison on Nonlinear (Throttled) Data

This trains four operator-learning architectures on the same data: the geometry1
thermal-throttling pilot (15 train / 5 test scenarios, 6/20 triggered derating, 90°C
threshold). The goal is to check whether a properly-resourced GPU run changes the
CPU-smoke-test result already on record (`docs/report.md` §9.7/§9.8, `goal.md` Track A3).

| architecture | conditioning | physics loss |
|---|---|---|
| `fno` | none (baseline) | off |
| `cond-fno` | FiLM on HTC/T_amb/TSV_frac | off |
| `cno-fno` | FiLM + CNO multiscale | on (PDE + flux) |
| `cno-fno` + attention | FiLM + CNO + axial self-attention (SAU-FNO style) | on (PDE + flux) |

Why this matters: every FNO run on this benchmark so far has been CPU-capacity-limited
(16-32 channels, 100-500 epochs on a 5-15 scenario CPU run). The CPU throttled-pilot run
already on record (plain FNO, 100 epochs, `--cpu-fast`) trailed ridge badly: det.MAE
1.795 K vs ridge's 0.707 K, spatial R² 0.551 vs ridge's 0.919. This notebook checks whether
more capacity plus the two richer architectures (CondFNO, SAU-style CNO-FNO) close that gap
on the one regime in this benchmark that is provably nonlinear (thermal throttling is a
closed feedback loop, power depends on the temperature field being solved for).

The bar to beat (already measured, `docs/report.md` §9.8, same data):

| baseline | det.MAE (K) | spatial R² | hotspot loc. err (µm) |
|---|---|---|---|
| ridge (nominal power) | 0.707 | 0.919 | 0 |
| kNN (k=3, nominal power) | 0.526 | 0.890 | n/a |
| FNO, CPU smoke test (plain, 100 ep) | 1.795 | 0.551 | 5139 |

If none of the four GPU-scale architectures beat ridge here, that's real (not
capacity-confounded) evidence for this project's central claim, on the one mechanism in
this benchmark designed specifically to break linearity. If one does, that's the strongest
counter-evidence available anywhere in this project and should be reported as such.

## Kaggle Dataset Setup

1. Source code: upload `src/` (slug suggestion: `thermo-pinn-src`, reuse if you already
   have it from other notebooks in this project).
2. Throttled pilot data: zip and upload the contents of
   `data/3d-ice-throttle-pilot/geometry1/` (the 20 `.npz` files, not the standard
   `data/3d-ice/` dataset, which has no throttled scenarios). Slug suggestion:
   `3dice-throttle-pilot-data`.
3. Enable GPU accelerator (Settings -> Accelerator -> GPU T4 x1).
4. Run all cells.

Expected time: about 3-8 min/model at 400 epochs on a T4 for this small (15-scenario)
dataset, under 30 min total for all four. Well inside the 30 GPU-hr/week free tier.


In [ ]:
import subprocess, sys

# Install any missing packages (PyYAML is the only non-standard dep)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)

import os
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: no GPU detected -- enable it under Settings -> Accelerator before running the training cells.')


In [ ]:
import sys
from pathlib import Path

# -- Kaggle dataset paths ----------------------------------------------------
# Adjust these slugs to match the datasets you attached to this notebook.
SRC_DATASET  = 'thermo-pinn-src'          # dataset containing the src/ folder
DATA_DATASET = '3dice-throttle-pilot-data'  # dataset containing the throttled geometry1 .npz files

SRC_ROOT  = Path(f'/kaggle/input/{SRC_DATASET}')
DATA_ROOT = Path(f'/kaggle/input/{DATA_DATASET}')
OUT_DIR   = Path('/kaggle/working/checkpoints/a3_throttled_arch_comparison')

sys.path.insert(0, str(SRC_ROOT))

assert SRC_ROOT.exists(),  f'Source dataset not found at {SRC_ROOT}. Check SRC_DATASET slug.'
assert DATA_ROOT.exists(), f'Data dataset not found at {DATA_ROOT}. Check DATA_DATASET slug.'
print('Source root:', SRC_ROOT)
print('Data root:  ', DATA_ROOT)


In [ ]:
import logging, time, json
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.core.geometry_builders import get_geometry_by_name
from src.pinn.data_loader import NormStats, compute_norm_stats
from src.fno.model import build_fno, build_cond_fno, build_cno_fno
from src.fno.data_loader import FNODataset, predict_to_flat
from src.fno.trainer import FNOTrainer

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
print('Imports OK')


In [ ]:
def metrics(pred: np.ndarray, true: np.ndarray, coords: np.ndarray) -> dict:
    """
    Same detrended-metric definition as scripts/baselines.py::metrics (not importable
    here); kept identical so results are comparable to the ridge/kNN numbers in docs/report.md.
    """
    err = pred - true
    ss_res = float(np.sum(err ** 2))
    ss_tot = float(np.sum((true - true.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')

    d_pred, d_true = pred - pred.mean(), true - true.mean()
    d_err = d_pred - d_true
    d_ss_res = float(np.sum(d_err ** 2))
    d_ss_tot = float(np.sum(d_true ** 2))
    spatial_r2 = 1.0 - d_ss_res / d_ss_tot if d_ss_tot > 0 else float('nan')

    hot_true_idx = int(np.argmax(true))
    hot_pred_idx = int(np.argmax(pred))
    hotspot_loc_err_um = float(np.linalg.norm(coords[hot_true_idx] - coords[hot_pred_idx]))
    hotspot_temp_err_K = float(pred[hot_true_idx] - true[hot_true_idx])

    return {
        'mae_K': float(np.mean(np.abs(err))),
        'rmse_K': float(np.sqrt(np.mean(err ** 2))),
        'max_abs_err_K': float(np.max(np.abs(err))),
        'r2': r2,
        'mae_detrended_K': float(np.mean(np.abs(d_err))),
        'spatial_r2': spatial_r2,
        'true_spatial_std_K': float(true.std()),
        'hotspot_temp_err_K': hotspot_temp_err_K,
        'hotspot_loc_err_um': hotspot_loc_err_um,
    }

print('metrics() defined')


In [ ]:
GEOM_NAME    = 'geometry1'
CHANNELS     = 32
N_BLOCKS     = 4
N_HEADS      = 4             # CNO-FNO axial attention heads; must divide CHANNELS evenly
MODES        = (16, 16, 12)  # fno/cond-fno spectral mode counts
EPOCHS       = 400           # same budget for all 4 models -- fair comparison, not each model's own default
BATCH_SIZE   = 4
LR           = 1e-3
PDE_WEIGHT   = 0.1           # only applied to the two cno-fno variants (physics=on)
FLUX_WEIGHT  = 0.5           # interface-isolated flux-continuity loss, cno-fno variants only

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')
print(f'Config: channels={CHANNELS} blocks={N_BLOCKS} epochs={EPOCHS}')


In [ ]:
geometry = get_geometry_by_name(GEOM_NAME)
geometries = {GEOM_NAME: geometry}
grid_shape = geometry.mesh_resolution
print(geometry.summary())
print('Grid shape (declared):', grid_shape)


In [ ]:
def collect_files(data_dir: Path, geom: str, split: str):
    files = sorted(data_dir.rglob(f'{geom}_{split}_*.npz'))
    if not files:
        files = sorted(data_dir.glob(f'{geom}_{split}_*.npz'))
    return files

train_files = collect_files(DATA_ROOT, GEOM_NAME, 'train')
test_files  = collect_files(DATA_ROOT, GEOM_NAME, 'test')

assert train_files, f'No training files found in {DATA_ROOT}. Check DATA_DATASET slug and file naming.'
assert test_files,  f'No test files found in {DATA_ROOT}.'
print(f'Training files : {len(train_files)}')
print(f'Test files     : {len(test_files)}')

# Confirm these are genuinely throttled scenarios, not the standard dataset by mistake.
_probe = np.load(train_files[0], allow_pickle=True)
_meta = dict(_probe['metadata'][0])
n_throttled = 0
for f in train_files + test_files:
    d = np.load(f, allow_pickle=True)
    if dict(d['metadata'][0]).get('throttle_enabled'):
        n_throttled += 1
print(f'throttle_enabled scenarios: {n_throttled}/{len(train_files) + len(test_files)}')
assert n_throttled > 0, 'No throttled scenarios found -- check DATA_DATASET points at the throttle pilot, not data/3d-ice/.'
assert 'power_nominal' in _probe.files, (
    'power_nominal missing from this .npz -- re-export with the current '
    'src/export/npz_exporter.py (goal.md Track A3 apples-to-apples fix) before uploading.'
)


In [ ]:
# Take the z-depth from the DATA, not geometry.mesh_resolution -- 3D-ICE emits one
# value per stack element (sub-layer discretisation), not the declared mesh nz.
# Mirrors scripts/train_fno.py's grid-from-data logic.
_d = np.load(train_files[0], allow_pickle=True)
_n = _d['coords'].shape[0]
_nx, _ny = grid_shape[0], grid_shape[1]
if _nx * _ny > 0 and _n % (_nx * _ny) == 0:
    data_grid = (_nx, _ny, _n // (_nx * _ny))
    if data_grid != tuple(grid_shape):
        print(f'Grid from data: {data_grid} (geometry declares {tuple(grid_shape)})')
        grid_shape = data_grid
print('Using grid_shape:', grid_shape)


In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
norm_path = OUT_DIR / 'norm_stats.json'

if norm_path.exists():
    norm_stats = NormStats.load(norm_path)
    print('Loaded existing norm stats from', norm_path)
else:
    print('Computing norm stats over', len(train_files), 'training files...')
    norm_stats = compute_norm_stats(train_files, geometries)
    norm_stats.save(norm_path)
    print('Saved norm stats to', norm_path)

print(f'T range: [{norm_stats.T_min:.1f}, {norm_stats.T_max:.1f}] K')


In [ ]:
print('Loading training dataset (FNODataset builds Q_norm from power_nominal automatically'
      ' when the field is present, per goal.md Track A3 -- same "requested power ->'
      ' resolved temperature" problem ridge is scored on)...')
train_dataset = FNODataset(train_files, norm_stats, grid_shape)
val_dataset   = FNODataset(test_files, norm_stats, grid_shape)

print(f'Train: {len(train_dataset)} scenarios')
print(f'Val  : {len(val_dataset)} scenarios')


In [ ]:
models = {
    'fno': build_fno(
        grid_shape=grid_shape, modes=MODES, hidden_ch=CHANNELS, n_blocks=N_BLOCKS, device=DEVICE,
    ),
    'cond-fno': build_cond_fno(
        grid_shape=grid_shape, modes=MODES, hidden_ch=CHANNELS, n_blocks=N_BLOCKS, device=DEVICE,
    ),
    'cno-fno': build_cno_fno(
        grid_shape=grid_shape, ch=CHANNELS, n_fno_blocks=N_BLOCKS,
        use_attention=False, device=DEVICE,
    ),
    'cno-fno+attn (SAU)': build_cno_fno(
        grid_shape=grid_shape, ch=CHANNELS, n_fno_blocks=N_BLOCKS,
        use_attention=True, n_heads=N_HEADS, device=DEVICE,
    ),
}

for name, m in models.items():
    print(f'{name:<22} {m.n_parameters:>12,} params')


In [ ]:
# physics loss only applies to the two cno-fno variants (matches _MODEL_DEFAULTS# in scripts/train_fno.py: physics on for cno-fno, off for fno/cond-fno)physics_on = {'fno': False, 'cond-fno': False, 'cno-fno': True, 'cno-fno+attn (SAU)': True}trainers = {}times = {}for name, model in models.items():    pde_w  = PDE_WEIGHT  if physics_on[name] else 0.0    flux_w = FLUX_WEIGHT if physics_on[name] else 0.0    out_dir = OUT_DIR / name.replace(' ', '_').replace('(', '').replace(')', '')    print(f'\n=== Training {name} (physics={physics_on[name]}) ===')    t0 = time.time()    trainer = FNOTrainer(        model=model, norm_stats=norm_stats, train_data=train_dataset, val_data=val_dataset,        output_dir=out_dir, batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, device=DEVICE,        geometry_name=f'{GEOM_NAME}_{name}', pde_weight=pde_w, flux_weight=flux_w,        geometry=geometry, log_interval=max(1, EPOCHS // 10),        use_amp=False,  # cuFFT half-precision needs power-of-two grids; ours never are    )    trainer.train()    elapsed = time.time() - t0    trainers[name] = trainer    times[name] = elapsed    print(f'{name}: best val MAE={trainer.best_val_mae:.3f} K, time={elapsed/60:.1f} min')

In [ ]:
# Evaluate every model on the test set with the SAME detrended metrics ridge/kNN
# were scored on (docs/report.md Sec 9.7/9.8), for a direct, apples-to-apples comparison.
nx, ny, nz = grid_shape
xs = np.linspace(0, geometry.die_width, nx)
ys = np.linspace(0, geometry.die_length, ny)
zs = np.linspace(0, geometry.get_total_height(), nz)
X, Y, Z = np.meshgrid(xs, ys, zs, indexing='ij')
coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)

results = {}
for name, model in models.items():
    model.eval()
    all_m = []
    with torch.no_grad():
        for item in val_dataset.items:
            T_pred_K, T_true_K = predict_to_flat(model, item, DEVICE, norm_stats)
            m = metrics(T_pred_K, T_true_K, coords)
            m['name'] = item['name']
            all_m.append(m)
    agg = {k: float(np.mean([m[k] for m in all_m]))
           for k in ('mae_K', 'mae_detrended_K', 'spatial_r2', 'hotspot_loc_err_um')}
    results[name] = agg
    print(name, agg)


In [ ]:
REFERENCE = {
    'ridge (nominal power)':        {'mae_detrended_K': 0.707, 'spatial_r2': 0.919, 'hotspot_loc_err_um': 0.0},
    'kNN, k=3 (nominal power)':     {'mae_detrended_K': 0.526, 'spatial_r2': 0.890, 'hotspot_loc_err_um': None},
    'FNO, CPU smoke test (plain)':  {'mae_detrended_K': 1.795, 'spatial_r2': 0.551, 'hotspot_loc_err_um': 5139.0},
}

print(f'{"model":<24} {"det.MAE (K)":>12} {"spatial R2":>12} {"hotspot err (um)":>18}')
print('-' * 70)
for name, r in REFERENCE.items():
    hot = f'{r["hotspot_loc_err_um"]:.0f}' if r['hotspot_loc_err_um'] is not None else '--'
    print(f'{name:<24} {r["mae_detrended_K"]:>12.3f} {r["spatial_r2"]:>12.3f} {hot:>18}')
print('-' * 70)
for name, r in results.items():
    beat_ridge = r['spatial_r2'] > REFERENCE['ridge (nominal power)']['spatial_r2']
    marker = '  <-- BEATS RIDGE' if beat_ridge else ''
    print(f'{name:<24} {r["mae_detrended_K"]:>12.3f} {r["spatial_r2"]:>12.3f} {r["hotspot_loc_err_um"]:>18.0f}{marker}')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = {'fno': 'gray', 'cond-fno': 'steelblue', 'cno-fno': 'darkorange', 'cno-fno+attn (SAU)': 'crimson'}
for name, trainer in trainers.items():
    ax.plot(trainer.history['epoch'], trainer.history['val_mae_K'], label=name, color=colors.get(name))
ax.axhline(REFERENCE['ridge (nominal power)']['mae_detrended_K'], color='black', linestyle='--',
           label='ridge det.MAE (reference, not directly comparable to raw val MAE)', alpha=0.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('Val MAE (K)')
ax.set_title(f'{GEOM_NAME} (throttled pilot): architecture comparison')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
summary = {
    'geometry': GEOM_NAME,
    'grid_shape': list(grid_shape),
    'epochs': EPOCHS,
    'channels': CHANNELS,
    'n_blocks': N_BLOCKS,
    'reference': REFERENCE,
    'results': results,
    'params': {name: m.n_parameters for name, m in models.items()},
    'train_time_min': {name: t / 60 for name, t in times.items()},
}

with open(OUT_DIR / 'a3_throttled_arch_comparison_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print()
print('Download this JSON plus the *_best.pt checkpoints from /kaggle/working/checkpoints/')
print('and report the results table back -- goal.md Track A3 / docs/report.md Sec 9.7-9.8')
print('are the two places this result needs to be recorded.')
